In [ ]:
!pip install trectools

In [2]:
import json
import random
import torch
import numpy as np
from tqdm import tqdm
from collections import defaultdict


from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
with open("joker_task1_retrieval_corpus25_EN.json") as f:
    corpus = json.load(f)

with open("joker_task1_retrieval_queries_train25_EN.json") as f:
    queries_train = json.load(f)

with open("joker_task1_retrieval_queries_test25_EN.json") as f:
    queries_test = json.load(f)

with open("joker_task1_retrieval_qrels_train25_EN.json") as f:
    qrels_train = json.load(f)

print(corpus[0], queries_train[0], qrels_train[0])

{'docid': '1', 'text': 'He has a green body, no visible nose, and lives in a trash can.'} {'qid': '8', 'query': 'colors'} {'qid': 8, 'docid': 151, 'qrel': 1}


In [ ]:
retriever = SentenceTransformer(
    "paraphrase-multilingual-mpnet-base-v2",
    device=DEVICE
)

In [6]:
doc_texts = [d["text"] for d in corpus]
doc_ids = [d["docid"] for d in corpus]

doc_embeddings = retriever.encode(
    doc_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    batch_size=256,
    show_progress_bar=True
)

Batches:   0%|          | 0/304 [00:00<?, ?it/s]

In [ ]:
clf_name = "xlm-roberta-base"
clf_tokenizer = AutoTokenizer.from_pretrained(clf_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(
    clf_name, num_labels=2
).to(DEVICE)

In [8]:
qid2query = {str(q["qid"]): q["query"] for q in queries_train}
docid2text = {str(d["docid"]): d["text"] for d in corpus}

qrels_by_qid = defaultdict(list)
for r in qrels_train:
    qrels_by_qid[str(r["qid"])].append(r)

train_pairs = []

NEGATIVE_RATIO = 1.5
SIM_THRESHOLD = 0.0

for qid, rels in qrels_by_qid.items():
    if qid not in qid2query:
        continue

    query_text = qid2query[qid]

    q_emb = retriever.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    sim_scores = np.dot(doc_embeddings, q_emb)
    mask = sim_scores > SIM_THRESHOLD
    retrieved_docids = {
        str(doc_ids[i]) for i in np.where(mask)[0]
    }

    positive_docids = {
        str(r["docid"]) for r in rels if r["qrel"] == 1
    }

    positives = [
        docid for docid in positive_docids
        if docid in retrieved_docids and docid in docid2text
    ]

    for docid in positives:
        train_pairs.append(
            (query_text, docid2text[docid], 1)
        )

    negatives = [
        docid for docid in retrieved_docids
        if docid not in positive_docids and docid in docid2text
    ]

    if len(positives) > 0 and len(negatives) > 0:
        sampled_negs = random.sample(
            negatives,
            min(len(negatives), int(NEGATIVE_RATIO * len(positives)))
        )

        for docid in sampled_negs:
            train_pairs.append(
                (query_text, docid2text[docid], 0)
            )


In [9]:
from torch.utils.data import Dataset, DataLoader

class PunDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=300):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, d, label = self.pairs[idx]

        enc = self.tokenizer(
            q,
            d,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = PunDataset(train_pairs, clf_tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

In [ ]:
from torch.optim import AdamW
from sklearn.metrics import f1_score

optimizer = AdamW(clf_model.parameters(), lr=2e-3)
clf_model.to(DEVICE)

EPOCHS = 5
best_f1 = 0.0

for epoch in range(EPOCHS):
    clf_model.train()
    all_preds, all_labels = [], []

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = clf_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds)
    print(f"Epoch {epoch+1} | Train F1 = {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(clf_model.state_dict(), "best_pun_model.pt")


In [ ]:
clf_model.load_state_dict(torch.load("best_pun_model.pt"))
clf_model.eval()

In [ ]:
def humor_scores_qd(queries, docs, batch_size=256):
    scores = []

    for i in tqdm(range(0, len(docs), batch_size)):
        batch_docs = docs[i:i+batch_size]
        batch_queries = queries[i:i+batch_size]

        batch_docs = [d if isinstance(d, str) else "" for d in batch_docs]
        batch_queries = [q if isinstance(q, str) else "" for q in batch_queries]

        enc = clf_tokenizer(
            batch_queries,
            batch_docs,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            logits = clf_model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]

        scores.extend(probs.cpu().numpy())

    return np.array(scores)

In [ ]:
SIM_THRESHOLD = 0.0

def retrieve(query, top_k=1000):
    q_emb = retriever.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    sim_scores = np.dot(doc_embeddings, q_emb)

    mask = sim_scores > SIM_THRESHOLD
    sem_idx = np.where(mask)[0]

    sem_idx = sem_idx[np.argsort(sim_scores[sem_idx])[::-1]]

    cand_docs = [doc_texts[i] for i in sem_idx]
    cand_queries = [query] * len(cand_docs)

    humor_qd = humor_scores_qd(cand_queries, cand_docs)

    order = np.argsort(humor_qd)[::-1][:top_k]

    return [{
        "docid": doc_ids[sem_idx[i]],
        "rank": r + 1,
        "score": float(humor_qd[i])
    } for r, i in enumerate(order)]

In [ ]:
train_predictions = []

for q in tqdm(queries_train):
    res = retrieve(q["query"], top_k=1000)
    for r in res:
        train_predictions.append({
            "qid": q["qid"],
            "docid": r["docid"],
            "rank": r["rank"],
            "score": r["score"]
        })

In [ ]:
with open("run_train.txt","w") as f:
    for r in train_predictions:
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {r['score']} run\n")

with open("qrels_train.txt","w") as f:
    for q in qrels_train:
        f.write(f"{q['qid']} 0 {q['docid']} {q['qrel']}\n")

In [ ]:
from trectools import TrecRun, TrecQrel, TrecEval

run = TrecRun("run_train.txt")
qrels = TrecQrel("qrels_train.txt")

ev = TrecEval(run, qrels)

metrics = {
    "map": ev.get_map(),
    "recip_rank": ev.get_reciprocal_rank(),

    "ndcg_5": ev.get_ndcg(5),
    "ndcg_10": ev.get_ndcg(10),
    "ndcg_20": ev.get_ndcg(20),

    "P_5": ev.get_precision(5),
    "P_10": ev.get_precision(10),
    "P_20": ev.get_precision(20),

    "recall_5": ev.get_recall(5),
    "recall_10": ev.get_recall(10),
    "recall_20": ev.get_recall(20),
}

metrics

In [ ]:
# predictions = []

# for q in tqdm(queries_test):
#     res = retrieve(q["query"], top_k=1000)
#     for r in res:
#         predictions.append({
#             "run_id": "run_test",
#             "manual": 0,
#             "qid": q["qid"],
#             "docid": r["docid"],
#             "rank": r["rank"],
#             "score": r["score"]
#         })

# with open("prediction.json","w") as f:
#     json.dump(predictions, f, indent=2)